Importacion de librerias iniciales y .env

In [ ]:
import os
from dotenv import load_dotenv
from google.cloud import compute_v1

In [ ]:
load_dotenv()

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
ZONE = os.getenv("GCP_ZONE_REGION", "us-west1-b")
INSTANCE_NAME = os.getenv("GCP_INSTANCE_NAME", "ssm-console-compute")
MACHINE_TYPE = os.getenv("GCP_MACHINE_TYPE", "e2-medium")
IMAGE_PROJECT = os.getenv("GCP_SOURCE_IMAGE_PROJECT", "ubuntu-os-cloud")
IMAGE_FAMILY = os.getenv("GCP_SOURCE_IMAGE_FAMILY", "ubuntu-2404-lts")

PG_USER = os.getenv("PG_USER", "admin_user")
PG_PASSWORD = os.getenv("PG_PASSWORD", "SuperSecurePassword123!")
PG_DB = os.getenv("PG_DB", "smartdata_db")

# Script de inicio desatendido para PostgreSQL
startup_script = f"""#!/bin/bash
set -e

apt-get update
apt-get install -y postgresql postgresql-contrib

# Permitir conexiones remotas
sed -i "s/#listen_addresses = 'localhost'/listen_addresses = '*'/g" /etc/postgresql/*/main/postgresql.conf
echo "host all all 0.0.0.0/0 md5" >> /etc/postgresql/*/main/pg_hba.conf

systemctl restart postgresql

# Crear DB y usuario
sudo -u postgres psql -c "CREATE USER {PG_USER} WITH PASSWORD '{PG_PASSWORD}';"
sudo -u postgres psql -c "CREATE DATABASE {PG_DB} OWNER {PG_USER};"
sudo -u postgres psql -c "GRANT ALL PRIVILEGES ON DATABASE {PG_DB} TO {PG_USER};"
"""

# Configurar especificación de la VM
instance = compute_v1.Instance()
instance.name = INSTANCE_NAME
instance.machine_type = f"zones/{ZONE}/machineTypes/{MACHINE_TYPE}"

disk = compute_v1.AttachedDisk()
disk.boot = True
disk.auto_delete = True
initialize_params = compute_v1.AttachedDiskInitializeParams()
initialize_params.source_image = f"projects/{IMAGE_PROJECT}/global/images/family/{IMAGE_FAMILY}"
initialize_params.disk_size_gb = 30
disk.initialize_params = initialize_params
instance.disks = [disk]

network_interface = compute_v1.NetworkInterface()
network_interface.name = "global/networks/default"
access_config = compute_v1.AccessConfig()
access_config.name = "External NAT"
access_config.type_ = compute_v1.AccessConfig.Type.ONE_TO_ONE_NAT.name
network_interface.access_configs = [access_config]
instance.network_interfaces = [network_interface]

instance.metadata = compute_v1.Metadata()
instance.metadata.items = [compute_v1.Items(key="startup-script", value=startup_script)]
instance.tags = compute_v1.Tags()
instance.tags.items = ["postgres-server", "http-server"]

# Desplegar
instances_client = compute_v1.InstancesClient()
operations_client = compute_v1.ZoneOperationsClient()

print(f"Desplegando VM '{INSTANCE_NAME}' en {ZONE}...")
operation = instances_client.insert(project=PROJECT_ID, zone=ZONE, instance_resource=instance)
result = operations_client.wait(project=PROJECT_ID, zone=ZONE, operation=operation.name)

if result.status == compute_v1.Operation.Status.DONE:
    instance_info = instances_client.get(project=PROJECT_ID, zone=ZONE, instance=INSTANCE_NAME)
    public_ip = instance_info.network_interfaces[0].access_configs[0].nat_i_p
    print(f"✓ VM creada exitosamente. IP Pública: {public_ip}")
    print(f"DSN PostgreSQL: postgresql://{PG_USER}:{PG_PASSWORD}@{public_ip}:5432/{PG_DB}")

Verificar autentificacion con GCP

In [ ]:
from google.colab import auth
auth.authenticate_user()

Importamos las librerías necesarias

In [ ]:
from __future__ import annotations
from collections.abc import Iterable
from google.cloud import compute_v1

In [ ]:
par_project_id = 'western-trilogy-429606-h1'

Crear instancia

In [ ]:
def wait_for_extended_operation(
    operation: ExtendedOperation, verbose_name: str = "operation", timeout: int = 300
) -> Any:

    result = operation.result(timeout=timeout)

    if operation.error_code:
        print(
            f"Error during {verbose_name}: [Code: {operation.error_code}]: {operation.error_message}",
            file=sys.stderr,
            flush=True,
        )
        print(f"Operation ID: {operation.name}", file=sys.stderr, flush=True)
        raise operation.exception() or RuntimeError(operation.error_message)

    if operation.warnings:
        print(f"Warnings during {verbose_name}:\n", file=sys.stderr, flush=True)
        for warning in operation.warnings:
            print(f" - {warning.code}: {warning.message}", file=sys.stderr, flush=True)

    return result

def create_instance(
    project_id: str, zone: str, instance_name: str, machine_type: str, source_image_project: str, source_image_family: str, network_name: str
) -> None:
    instance_client = compute_v1.InstancesClient()

    # Configure the machine type
    machine_type_full_path = f"zones/{zone}/machineTypes/{machine_type}"

    # Create the boot disk with the correct source image
    disk = compute_v1.AttachedDisk(
        initialize_params=compute_v1.AttachedDiskInitializeParams(
            source_image=f"projects/{source_image_project}/global/images/family/{source_image_family}"
        ),
        auto_delete=True,
        boot=True,
        type_=compute_v1.AttachedDisk.Type.PERSISTENT.name,
    )

    # Configure the network interface
    network_interface = compute_v1.NetworkInterface(name=network_name)

    # Define the instance resource
    instance = compute_v1.Instance(
        name=instance_name,
        machine_type=machine_type_full_path,
        disks=[disk],
        network_interfaces=[network_interface],
    )

    print(f"Creating instance {instance_name} in {zone}...")
    operation = instance_client.insert(project=project_id, zone=zone, instance_resource=instance)
    wait_for_extended_operation(operation, "instance creation")
    print(f"Instance {instance_name} created.")


In [ ]:
create_instance(
    project_id=par_project_id,
    zone="us-central1-f",
    instance_name="sd-colab-vm-02",
    machine_type="n1-standard-1",
    source_image_project="debian-cloud",  # Project hosting the image
    source_image_family="debian-11",      # Updated to Debian 11
    network_name="global/networks/default"
)

Creating instance sd-colab-vm-02 in us-central1-f...
Instance sd-colab-vm-02 created.


Listar las instancias de Compute Engine por Zona

In [ ]:
def list_instances(project_id: str, zone: str) -> Iterable[compute_v1.Instance]:
    instance_client = compute_v1.InstancesClient()
    instance_list = instance_client.list(project=project_id, zone=zone)

    print(f"Instances found in zone {zone}:")
    for instance in instance_list:
        print(f" - {instance.name} ({instance.machine_type})")

    return instance_list

In [ ]:
instancias = list_instances(par_project_id,'us-central1-f')

Instances found in zone us-central1-f:
 - sd-colab-vm-01 (https://www.googleapis.com/compute/v1/projects/western-trilogy-429606-h1/zones/us-central1-f/machineTypes/n1-standard-1)
 - sd-colab-vm-02 (https://www.googleapis.com/compute/v1/projects/western-trilogy-429606-h1/zones/us-central1-f/machineTypes/n1-standard-1)


In [ ]:
lista_instancias = list(instancias)

for instance in lista_instancias:
    with open("my_file.txt", "w") as archivo:
    # Escribimos los datos en el archivo
        archivo.write(str(instance))

In [ ]:
!pwd

/content


Detener una instancia de Compute Engine

In [ ]:
from __future__ import annotations

import sys
from typing import Any

from google.api_core.extended_operation import ExtendedOperation
from google.cloud import compute_v1

def stop_instance(project_id: str, zone: str, instance_name: str) -> None:
    instance_client = compute_v1.InstancesClient()

    operation = instance_client.stop(
        project=project_id, zone=zone, instance=instance_name
    )
    wait_for_extended_operation(operation, "instance stopping")


In [ ]:
stop_instance(par_project_id, 'us-central1-f', 'sd-colab-vm-01')

Iniciar una instancia de Compute Engine

In [ ]:
from __future__ import annotations

import sys
from typing import Any

from google.api_core.extended_operation import ExtendedOperation
from google.cloud import compute_v1

def wait_for_extended_operation(
    operation: ExtendedOperation, verbose_name: str = "operation", timeout: int = 300
) -> Any:

    result = operation.result(timeout=timeout)

    if operation.error_code:
        print(
            f"Error during {verbose_name}: [Code: {operation.error_code}]: {operation.error_message}",
            file=sys.stderr,
            flush=True,
        )
        print(f"Operation ID: {operation.name}", file=sys.stderr, flush=True)
        raise operation.exception() or RuntimeError(operation.error_message)

    if operation.warnings:
        print(f"Warnings during {verbose_name}:\n", file=sys.stderr, flush=True)
        for warning in operation.warnings:
            print(f" - {warning.code}: {warning.message}", file=sys.stderr, flush=True)

    return result

def start_instance(project_id: str, zone: str, instance_name: str) -> None:

    instance_client = compute_v1.InstancesClient()

    operation = instance_client.start(
        project=project_id, zone=zone, instance=instance_name
    )

    wait_for_extended_operation(operation, "instance start")


In [ ]:
start_instance(par_project_id, 'us-central1-f', 'sd-colab-vm-01')

Elimina una instancia de Compute Engine

In [ ]:
from __future__ import annotations

import sys
from typing import Any

from google.api_core.extended_operation import ExtendedOperation
from google.cloud import compute_v1

def wait_for_extended_operation(
    operation: ExtendedOperation, verbose_name: str = "operation", timeout: int = 300
) -> Any:

    result = operation.result(timeout=timeout)

    if operation.error_code:
        print(
            f"Error during {verbose_name}: [Code: {operation.error_code}]: {operation.error_message}",
            file=sys.stderr,
            flush=True,
        )
        print(f"Operation ID: {operation.name}", file=sys.stderr, flush=True)
        raise operation.exception() or RuntimeError(operation.error_message)

    if operation.warnings:
        print(f"Warnings during {verbose_name}:\n", file=sys.stderr, flush=True)
        for warning in operation.warnings:
            print(f" - {warning.code}: {warning.message}", file=sys.stderr, flush=True)

    return result

def delete_instance(project_id: str, zone: str, machine_name: str) -> None:

    instance_client = compute_v1.InstancesClient()

    print(f"Deleting {machine_name} from {zone}...")
    operation = instance_client.delete(
        project=project_id, zone=zone, instance=machine_name
    )
    wait_for_extended_operation(operation, "instance deletion")
    print(f"Instance {machine_name} deleted.")


In [ ]:
delete_instance(par_project_id, 'us-central1-f', 'sd-colab-vm-01')

Deleting sd-colab-vm-01 from us-central1-f...
Instance sd-colab-vm-01 deleted.
